In [ ]:
import numpy as np

# ── 1. 기초 데이터 및 제약 조건 설정 ───────────────────────────
CROPS = ['양상추','토마토','대파','오이','고추','시금치','무','배추','옥수수']
N_CROP = 9
N_PERIOD = 24

area_per = np.array([0.10, 0.30, 0.15, 0.40, 0.25, 0.05, 0.20, 0.20, 0.20])
grow_O = np.array([4, 8, 6, 5, 10, 3, 6, 5, 6])
grow_H = np.array([3, 6, 5, 4,  8, 2, 5, 4, 5])

price_O_monthly = np.array([
    [2500,2500,2500,2500,2500,   0,   0,   0,   0,   0,2200,2200],
    [   0,   0,   0,   0,15000,18000,18000,15000,12000,  0,  0,  0],
    [   0,   0,4000,4000,4500,5000,5000,5500,5000,4500,  0,  0],
    [   0,   0,   0,   0,11000,11000,13000,15000,13000,11000,0, 0],
    [   0,   0,   0,   0,   0,   0,10000,12000,12000,10000,  0,  0],
    [ 800, 800, 600, 600,   0,   0,   0,   0, 700, 700, 800, 800],
    [   0,   0,   0,   0,   0,   0,   0,   0,4500,4800,4800,4800],
    [   0,   0,   0,   0,   0,   0,   0,   0,3200,3200,3600,3600],
    [   0,   0,   0,   0,   0,   0,3000,3500,3000,   0,   0,   0],
])
price_H_monthly = np.array([
    [3500,3500,3200,3200,3200,3000,3000,3500,3500,3200,3200,3200],
    [   0,   0,25000,25000,21000,25000,25000,21000,18000,20000,22000,22000],
    [6000,6000,6000,6000,6500,7000,7000,8000,7000,6500,7000,7000],
    [   0,   0,15000,15000,15000,16000,18000,21000,18000,16000,18000,18000],
    [   0,   0,15000,15000,15000,15000,14000,17000,17000,15000,18000,18000],
    [1200,1200,1000,1000,1000,1000,1000,1200,1000,1000,1200,1200],
    [6500,6500,6000,6000,6000,6000,6000,6500,6000,6500,6500,6500],
    [5500,5500,5000,5000,5000,5000,5000,5500,4800,4800,5000,5000],
    [   0,   0,   0,   0,4500,4500,4200,5000,4200,4500,   0,   0],
])

AREA_O = 1200.0;  AREA_H = 300.0
MAX_O  =  300.0;  MAX_H  =  75.0
MIN_O  =   60.0;  MIN_H  =  15.0

# ── 2. 유효 슬롯 필터링 및 단위수익 계산 ────────────────────────
valid_O = np.zeros((N_CROP, N_PERIOD), dtype=bool)
valid_H = np.zeros((N_CROP, N_PERIOD), dtype=bool)
rev_O   = np.zeros((N_CROP, N_PERIOD))
rev_H   = np.zeros((N_CROP, N_PERIOD))

for i in range(N_CROP):
    for t in range(N_PERIOD):
        ht_O = t + grow_O[i] - 1
        ht_H = t + grow_H[i] - 1
        if ht_O < N_PERIOD:
            p = price_O_monthly[i, ht_O // 2]
            if p > 0:
                valid_O[i, t] = True
                rev_O[i, t]   = p / area_per[i]
        if ht_H < N_PERIOD:
            p = price_H_monthly[i, ht_H // 2]
            if p > 0:
                valid_H[i, t] = True
                rev_H[i, t]   = p / area_per[i]

# ── 3. 후보 수집 ─────────────────────────────────────────────────
candidates = []
for t in range(N_PERIOD):
    for i in range(N_CROP):
        if valid_O[i, t]:
            candidates.append((rev_O[i, t], grow_O[i], i, t, 'O'))
        if valid_H[i, t]:
            candidates.append((rev_H[i, t], grow_H[i], i, t, 'H'))

In [ ]:
def quicksort(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    pk = (-pivot[0], pivot[1], pivot[2])
    left  = [x for x in arr if (-x[0], x[1], x[2]) < pk]
    mid   = [x for x in arr if (-x[0], x[1], x[2]) == pk]
    right = [x for x in arr if (-x[0], x[1], x[2]) > pk]
    return quicksort(left) + mid + quicksort(right)

candidates = quicksort(candidates)

In [ ]:
# ── 4. 그리디 배정 ──────────────────────────────────────────────
used_O = np.zeros(N_PERIOD)
used_H = np.zeros(N_PERIOD)
xO = np.zeros((N_CROP, N_PERIOD))
xH = np.zeros((N_CROP, N_PERIOD))
occ_O = np.zeros((N_CROP, N_PERIOD))
occ_H = np.zeros((N_CROP, N_PERIOD))

for rev, grow_time, i, t, land in candidates:
    if land == 'O':
        end_t = t + grow_time
        if end_t > N_PERIOD: continue
        available = AREA_O - used_O[t:end_t].max()
        alloc = min(MAX_O, available)
        if alloc >= MIN_O:
            xO[i, t] = alloc
            used_O[t:end_t] += alloc
            occ_O[i, t:end_t] += alloc
    else:
        end_t = t + grow_time
        if end_t > N_PERIOD: continue
        available = AREA_H - used_H[t:end_t].max()
        alloc = min(MAX_H, available)
        if alloc >= MIN_H:
            xH[i, t] = alloc
            used_H[t:end_t] += alloc
            occ_H[i, t:end_t] += alloc

# ── 5. 결과 계산 및 출력 ────────────────────────────────────────
total_rev = (rev_O * xO + rev_H * xH).sum()
crop_rev  = np.array([(rev_O[i]*xO[i] + rev_H[i]*xH[i]).sum() for i in range(N_CROP)])

print("=" * 52)
print(f"  그리디 최적해: {total_rev/1e6:.4f} 백만원")
print("=" * 52)
print(f"\n{'작물':<6} {'수익(백만원)':>12} {'노지(m2)':>10} {'하우스(m2)':>10}")
print("-" * 42)
for i in range(N_CROP):
    print(f"{CROPS[i]:<6} {crop_rev[i]/1e6:>12.3f} {xO[i].sum():>10.1f} {xH[i].sum():>10.1f}")
print(f"\n  합계  {total_rev/1e6:>12.4f} {xO.sum():>10.1f} {xH.sum():>10.1f}")